Parcial #2 - Electiva 1 14/Oct/2026
Presentado por: Jonhatan Stiven Gutiérrez Vélez
El código es elaboración propia con ayuda de IA para la verificación y corrección de algunos errores.

1. Cáculo de PET, PEA, PEI, TO, TD a nivel nacional

In [0]:
#A. Población en edad de trabajar
import pandas as pd

#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

In [0]:
#A. POBLACION EN EDAD DE TRABAJAR
PET = ft[ft['PET']==1]['FEX_C18'].sum()
PET
print(f"A nivel nacional, Colombia cuenta con {PET:,.0f} personas en Edad de Trabajar (PET).")

In [0]:
#B. POBLACION QUE NO ESTA EN EDAD DE TRABAJAR
no_PET = cgse[cgse['P6040'] < 15]['FEX_C18'].sum()
no_PET
print(f"A nivel nacional, Colombia cuenta con {no_PET:,.0f} personas que NO están en Edad de Trabajar (menores de 15 años).")

In [0]:
#C. POBLACION ECONOMICAMENTE ACTIVA
PEA = ft[ft['FT']==1]['FEX_C18'].sum()
PEA
print(f"A nivel nacional, Colombia cuenta con {PEA:,.0f} personas Económicamente Activas (PEA).")

In [0]:
#D. POBLACION ECONOMICAMENTE INACTIVA
PEI = PET - PEA
print(f"A nivel nacional, Colombia cuenta con {PEI:,.0f} personas Económicamente Inactivas (PEI).")

In [0]:
#E. TASA DE OCUPACION
ocupados_total = ocupados['FEX_C18'].sum()
TO = (ocupados_total / PET) * 100
print(f"A nivel nacional, la Tasa de Ocupacion (TO) es del {TO:.1f}%.")

In [0]:
#F. TASA DE DESEMPLEO
D = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()
TD = (D / PEA) * 100
print(f"A nivel nacional, la Tasa de Desempleo (TD) es del {TD:.1f}%.")

A nivel nacional, los resultados muestran que Colombia cuenta con una Población en Edad de Trabajar (PET) de 41,202,631 personas, lo que representa el 78% de la población total (52.5 millones sumando los 11,346,181 menores de 15 años que no están en edad de trabajar). La Población Económicamente Activa (PEA) es de 26,593,429 personas, es decir, solo el 64.5% de la PET participa activamente en el mercado laboral, mientras que el restante 35.5% (14,609,202 personas) forma la Población Económicamente Inactiva (PEI): estudiantes, personas dedicadas a oficios del hogar, pensionados, entre otros.
La Tasa de Ocupación (TO) es del 59.4%, lo que indica que de cada 100 personas en edad de trabajar, aproximadamente 59 están ocupadas. Por su parte, la Tasa de Desempleo (TD) es del 8.0%, es decir, de cada 100 personas en la PEA, 8 buscan empleo activamente y no lo encuentran. Esta TD es moderada y se encuentra dentro del rango histórico de Colombia.

2. Cálculo de TO para hombres y mujeres para todos los departamentos

In [0]:
##2 #Leyendo la tabla de caracteristicas generales
ft = spark.read.format("delta").table("workspace.electiva.ft")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()

#Primero hay que anexar la p3271 a la tabla de desempleados
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:

desempleados

In [0]:
# desempleados_dpto -> cantidad de hombres y mujeres desempleados por departamento
desempleados_dpto_genero = desempleados[desempleados['DSI']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

In [0]:
desempleados_dpto_genero

In [0]:

#cantidad de hombres y mujeres economicamente activos por departamento
pea_dpto_genero = ft[ft['FT']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

In [0]:
pea_dpto_genero

In [0]:

tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero, 'DESEMPLEADOS':desempleados_dpto_genero})

In [0]:

tabla_desempleo

In [0]:
tabla_desempleo['TD']= (tabla_desempleo['DESEMPLEADOS']/tabla_desempleo['PEA'])*100

In [0]:
tabla_desempleo.head(100)

In [0]:

# 4. Diccionarios de mapeo
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO', 66: 'RISARALDA',
    68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA', 76: 'VALLE DEL CAUCA', 81: 'ARAUCA',
    85: 'CASANARE', 86: 'PUTUMAYO', 88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA',
    95: 'GUAVIARE', 97: 'VAUPES', 99: 'VICHADA'
}

sexo_nombre = {1:'Hombre',2:'Mujer'}
tabla_desempleo = tabla_desempleo.reset_index()

tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo_nombre)

In [0]:
tabla_desempleo = tabla_desempleo[['Departamento','Sexo','PEA','DESEMPLEADOS','TD']]
tabla_desempleo

In [0]:
tabla_desempleo = tabla_desempleo[['Departamento','Sexo','PEA','DESEMPLEADOS','TD']]
tabla_desempleo

In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(by=[ 'Sexo','TD'], ascending=False)
tabla_desempleo

In [0]:
tabla_desempleo[tabla_desempleo['Sexo']=='Hombre']

In [0]:
tabla_desempleo[tabla_desempleo['Sexo']=='Mujer']

Como podemos observar, en casi todos los departamentos, la tasa de desempleo en mujeres es superior a la de los hombres. Por ejemplo Bolívar, la TD en mujeres es de 17.4%, mientras que en los hombres es de 6.44%, casi el triple, y así se repite en la mayoría de departamentos. Esto genera mayores barreras de inserción laboral para las mujeres, y mi principal hipótesis es que al contratar una mujer, los costos son mayores para las empresas (debido a que generalmente las mujeres están al cuidado de familiares adultos mayores o de los hijos).
También podemos observar que los departamentos de la periferia concentran las TD más altas. Mieentras que los departamentos mas centralizados poseen las TD más bajas debido al dinamismo de la economía (como el Eje Cafetero, Antioquia, Bogotá y Valle del Cauca). Además, esto se explica por la Teoría de discriminación estadística desarrollada por Kenneth Arrow en 1970.

3. Cálculo de TO para diferentes grupos de edad para todos los departamentos

In [0]:
import pandas as pd
import numpy as np

dptos = {5: 'Antioquia', 8: 'Atlántico', 11: 'Bogotá', 13: 'Bolívar', 15: 'Boyacá', 17: 'Caldas', 18: 'Caquetá', 19: 'Cauca', 20: 'Cesar', 23: 'Córdoba', 25: 'Cundinamarca', 27: 'Chocó', 41: 'Huila', 44: 'La Guajira', 47: 'Magdalena', 50: 'Meta', 52: 'Nariño', 54: 'Norte de Santander', 63: 'Quindío', 66: 'Risaralda', 68: 'Santander', 70: 'Sucre', 73: 'Tolima', 76: 'Valle del Cauca', 81: 'Arauca', 85: 'Casanare', 86: 'Putumayo', 88: 'San Andrés y Providencia', 91: 'Amazonas', 94: 'Guainía', 95: 'Guaviare', 97: 'Vaupés', 99: 'Vichada'}

# traigo la edad y el DSI
llaves = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR']
ft = ft.drop(columns=['P6040', 'DSI', 'Grupo_Edad'], errors='ignore')
ft = ft.merge(cgse[llaves + ['P6040']], on=llaves, how='left')
ft = ft.merge(desempleados[llaves + ['DSI']], on=llaves, how='left')

# grupos de edad
edad = ft['P6040']
condiciones_modelo = [(edad >= 18) & (edad < 24),(edad >= 24) & (edad < 30),(edad >= 30) & (edad < 40),(edad > 40) & (edad <= 50),(edad > 60)]
grupos = ['18 a 24', '24 a 30', '30 a 40', '40 a 50', '60 en adelante']

ft['Grupo_Edad'] = np.select(condiciones_modelo, grupos, default='Otro')
datos = ft[ft['Grupo_Edad'] != 'Otro'].copy()
datos['Departamento'] = datos['DPTO'].map(dptos)

# PEA y desempleados por departamento y grupo
pea = datos[datos['FT'] == 1].groupby(['Departamento', 'Grupo_Edad'])['FEX_C18'].sum()
desocupados = datos[datos['DSI'] == 1].groupby(['Departamento', 'Grupo_Edad'])['FEX_C18'].sum()

tasa = (desocupados / pea * 100).fillna(0)

# Resultados por departamento (lista)
for depto in tasa.index.get_level_values(0).unique():
    print(f"\n{depto.upper()}")
    t = tasa[depto]
    for grupo in grupos:
        if grupo in t.index:
            print(f"  Tasa de desempleo {grupo} años: {t[grupo]:.2f}%")
    print(f"  ➤ Mayor TD: {t.idxmax()} años ({t.max():.2f}%)")
    print(f"  ➤ Menor TD: {t.idxmin()} años ({t.min():.2f}%)")

En la mayoría de los casos, el grupo de 60 años en adelante tiene la TD más baja, mientas que en el grupo entre 18 a 24 años posee la TD más alta, esto debido a que en ese momento, este grupo de personas están estudiando y no están interesadas en buscar empleo. Por ejemplo, la mayor TD está en Putumayo (24-30 años), está en 30.58%.
En la mayoría de departamentos, la TD disminuye a medida que aumenta la edad. Por ejemplo, para Bolívar, entre 18-24 años, la TD es de 30.86%; entre 24-30 años, la TD está en 17.17%; para el grupo 30-40 años, la TD está en 11.35%; para el grupo de 40-50 años, la TD es de 5.35%; y por último, para el grupo de 60 años en adelante, la TD está en 2.35%. Con base en estos resultados, se comportan también la mayoría de departamentos. Esto confirma que la experiencia y la estabilidad laboral aumentan con la edad, reduciendo así el riesgo a quedar desempleado.


4. Cálculo de los 3 departamentos con menor TD en mujeres entre 18 y 25 años

In [0]:
import pandas as pd

# Traer la columna P3271 (sexo) desde cgse hacia ft
llaves = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR']
ft = ft.drop(columns=['P3271'], errors='ignore')
ft = ft.merge(cgse[llaves + ['P3271']], on=llaves, how='left')

# 1. Calcular la PEA de mujeres entre 18 y 25 años por departamento
# Se filtra por edad (P6040), sexo femenino (P3271 == 2) y fuerza de trabajo (FT == 1)
pea_muj_18_25 = ft[ (ft['P6040'] >= 18) & (ft['P6040'] <= 25) & (ft['P3271'] == 2) & (ft['FT'] == 1)].groupby('DPTO')['FEX_C18'].sum()

# 2. Calcular la cantidad de mujeres desempleadas entre 18 y 25 años por departamento
# Se filtra por edad, sexo femenino y desocupados (DSI == 1)
# (Nota: verifica si tu variable de desocupados es DSI u otra que haya definido el profesor como D)
desemp_muj_18_25 = ft[(ft['P6040'] >= 18) &  (ft['P6040'] <= 25) & (ft['P3271'] == 2) & (ft['DSI'] == 1)].groupby('DPTO')['FEX_C18'].sum()

# 3. Calcular la tasa de desempleo siguiendo la fórmula del profesor: (Desempleados / PEA) * 100
tasa_desempleo_muj = (desemp_muj_18_25 / pea_muj_18_25) * 100

# 4. Ordenar los valores de menor a mayor y extraer los 3 primeros
t_3_menor_td = tasa_desempleo_muj.sort_values(ascending=True).head(3)

# cambio los códigos por el nombre del departamento
t_3_menor_td.index = t_3_menor_td.index.map(dptos)

print("Los 3 departamentos con menor tasa de desempleo para mujeres entre 18 y 25 años son:")
for depto, tasa in t_3_menor_td.items():
    print(f"  {depto}: {tasa:.2f}%")

Para este caso, debemos analizarlo regionalmente, ya que Nariño y Huila son departamentos de economías agrarias con alta informalidad. Por lo tanto, considero que es normal que en este grupo de edad, los jóvenes ayuden a sus familias a generar ingresos en el campo, pero por tener tasas de desempleo tan bajas, no necesariaente sea empleo de calidad, sino, empleo informal. Por el lado de Amazonas, es un departamento amplio pero poco habitado. Por lo tanto, muchas de esas personas migran a departamentos con mayor interacción económica.

5. Cálculo de promedio de INGLABO para Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca. 

In [0]:

# Traer la columna INGLABO (ingresos laborales) desde ocupados hacia ft
llaves = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR']
ft = ft.drop(columns=['INGLABO'], errors='ignore')
ft = ft.merge(ocupados[llaves + ['INGLABO']], on=llaves, how='left')

# 1. Promedio de ingresos laborales en Risaralda (filtrando quienes reportan ingresos > 0)
ft_risaralda = ft[(ft['DPTO']==66) & (ft['INGLABO']>0)]
prom_inglabo_ris = (ft_risaralda['INGLABO'] * ft_risaralda['FEX_C18']).sum() / ft_risaralda['FEX_C18'].sum()
print(f"El promedio de ingresos laborales en Risaralda es {prom_inglabo_ris:,.0f} PESOS")

# 2. Promedio de ingresos laborales en Caldas
ft_caldas = ft[(ft['DPTO']==17) & (ft['INGLABO']>0)]
prom_inglabo_cal = (ft_caldas['INGLABO'] * ft_caldas['FEX_C18']).sum() / ft_caldas['FEX_C18'].sum()
print(f"El promedio de ingresos laborales en Caldas es {prom_inglabo_cal:,.0f} PESOS")

# 3. Promedio de ingresos laborales en Quindio
ft_quindio = ft[(ft['DPTO']==63) & (ft['INGLABO']>0)]
prom_inglabo_qui = (ft_quindio['INGLABO'] * ft_quindio['FEX_C18']).sum() / ft_quindio['FEX_C18'].sum()
print(f"El promedio de ingresos laborales en Quindío es {prom_inglabo_qui:,.0f} PESOS")

# 4. Promedio de ingresos laborales en Antioquia
ft_antioquia = ft[(ft['DPTO']==5) & (ft['INGLABO']>0)]
prom_inglabo_ant = (ft_antioquia['INGLABO'] * ft_antioquia['FEX_C18']).sum() / ft_antioquia['FEX_C18'].sum()
print(f"El promedio de ingresos laborales en Antioquia es {prom_inglabo_ant:,.0f} PESOS")

# 5. Promedio de ingresos laborales en Valle del Cauca
ft_valle = ft[(ft['DPTO']==76) & (ft['INGLABO']>0)]
prom_inglabo_val = (ft_valle['INGLABO'] * ft_valle['FEX_C18']).sum() / ft_valle['FEX_C18'].sum()
print(f"El promedio de ingresos laborales en Valle del Cauca es {prom_inglabo_val:,.0f} PESOS")

Para este caso, Antioquia lidera en el promedio de ingresos con 2.49 millones de pesos.
Por otro lado, el Eje Cafetero tienen promedios similares, entre 2.17 a 2.23 millones de pesos, esto tiene coherencia con respecto a sus economías, ya que las 3 tienen su estructura productiva basadas en café y agricultura en general.
Por parte del Valle del Cauca, es el que tiene el promedio de ingresos más bajo que el resto de departamentos. Esto se podría explicar a que el Valle, posee una cantidad alta de personas trabajando desde la informalidad, además, muchos municipios tienen también empleos rurales, eso genera que el promedio de ingresos se reduzca de manera considerable.